# [Introduction to BERTopic Embeddings](https://maartengr.github.io/BERTopic/getting_started/embeddings/embeddings.html)

In BERTopic, **embeddings** play a crucial role in transforming raw text into meaningful numerical representations that can be clustered into topics. By default, BERTopic uses **`all-MiniLM-L6-v2`** from `sentence-transformers`, but it allows users to integrate various embedding methods depending on the project requirements. The choice of embeddings impacts the quality of topic separation, coherence, and interpretability.

---

## **Choosing the Right Embedding Approach for Different Projects**
| **Embedding Approach** | **When to Use It?** | **Example Use Case** |
|----------------------|------------------|----------------|
| **Sentence-Transformers (`all-MiniLM-L6-v2`)** | Best for general-purpose topic modeling with sentence-level context. | News articles, research papers, long-form content. |
| **Hugging Face Transformers (`bert-base-uncased`)** | When fine-tuning transformer models for domain-specific text is needed. | Medical, legal, or scientific topic modeling. |
| **Flair** | When using character-level and contextual word embeddings (LSTM-based models). | Analyzing informal text like social media posts, tweets, or legal case studies. |
| **spaCy** | When lightweight and efficient word embeddings are required. | Processing large datasets with simple linguistic structures (e.g., customer feedback). |
| **Universal Sentence Encoder (USE)** | When semantic similarity is crucial but computational efficiency is needed. | Chatbot logs, customer support messages, FAQ document classification. |
| **Gensim (`Word2Vec`, `FastText`)** | When working with **custom-trained** word embeddings for specialized vocabularies. | Finance, biomedical research, niche domain corpora. |
| **Scikit-learn (`TF-IDF` or `CountVectorizer`)** | When interpretability and bag-of-words-based modeling is required. | Small datasets, keyword-based topic modeling, legal documents. |
| **OpenAI Embeddings (`text-embedding-ada-002`)** | When high-quality, state-of-the-art embeddings are needed but cost isn't an issue. | AI-powered document retrieval, advanced NLP applications. |
| **Cohere Embeddings** | When using third-party APIs for enterprise-scale semantic search. | Large-scale business intelligence, recommendation systems. |
| **Multimodal (Text + Images)** | When dealing with mixed-media content (text, images, audio). | News clustering, academic research combining text and figures. |
| **Custom Embeddings** | When none of the standard embeddings fit the use case. | Proprietary models, embeddings from domain-specific sources. |

By selecting the right embedding approach, **BERTopic can be optimized for different domains, dataset sizes, and computational constraints**. 🚀

## Sentence Transformers

You can select any model from sentence-transformers here and pass it through BERTopic with embedding_model:

In [1]:
from bertopic import BERTopic
topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2")

Or select a SentenceTransformer model with your parameters:

In [2]:
from sentence_transformers import SentenceTransformer

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=sentence_model)

## 🤗 Hugging Face Transformers

To use a Hugging Face transformers model, load in a pipeline and point to any model found on their model hub (https://huggingface.co/models):

In [3]:
from transformers.pipelines import pipeline

embedding_model = pipeline("feature-extraction", model="distilbert-base-cased")
topic_model = BERTopic(embedding_model=embedding_model)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Device set to use mps:0


## Flair

Flair allows you to choose almost any embedding model that is publicly available. Flair can be used as follows:




In [4]:
from flair.embeddings import TransformerDocumentEmbeddings

roberta = TransformerDocumentEmbeddings('roberta-base')
topic_model = BERTopic(embedding_model=roberta)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

You can select any 🤗 transformers model here.

Moreover, you can also use Flair to use word embeddings and pool them to create document embeddings. Under the hood, Flair simply averages all word embeddings in a document. Then, we can easily pass it to BERTopic to use those word embeddings as document embeddings:

In [ ]:
from flair.embeddings import WordEmbeddings, DocumentPoolEmbeddings

glove_embedding = WordEmbeddings('crawl')
document_glove_embeddings = DocumentPoolEmbeddings([glove_embedding])

topic_model = BERTopic(embedding_model=document_glove_embeddings)

## Spacy

Spacy is an amazing framework for processing text. There are many models available across many languages for modeling text.

To use Spacy's non-transformer models in BERTopic:

In [ ]:
import spacy

nlp = spacy.load("en_core_web_md", exclude=['tagger', 'parser', 'ner', 
                                            'attribute_ruler', 'lemmatizer'])

topic_model = BERTopic(embedding_model=nlp)

Using spacy-transformer models:

In [ ]:
import spacy

spacy.prefer_gpu()
nlp = spacy.load("en_core_web_trf", exclude=['tagger', 'parser', 'ner', 
                                             'attribute_ruler', 'lemmatizer'])

topic_model = BERTopic(embedding_model=nlp)

If you run into memory issues with spacy-transformer models, try:

In [ ]:
import spacy
from thinc.api import set_gpu_allocator, require_gpu

nlp = spacy.load("en_core_web_trf", exclude=['tagger', 'parser', 'ner', 
                                             'attribute_ruler', 'lemmatizer'])
set_gpu_allocator("pytorch")
require_gpu(0)

topic_model = BERTopic(embedding_model=nlp)


## Universal Sentence Encoder (USE)

The Universal Sentence Encoder encodes text into high-dimensional vectors that are used here for embedding the documents. The model is trained and optimized for greater-than-word length text, such as sentences, phrases, or short paragraphs.

Using USE in BERTopic is rather straightforward:

In [ ]:
import tensorflow_hub
embedding_model = tensorflow_hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")
topic_model = BERTopic(embedding_model=embedding_model)

## Gensim

BERTopic supports the gensim.downloader module, which allows it to download any word embedding model supported by Gensim. Typically, these are Glove, Word2Vec, or FastText embeddings:

In [ ]:
import gensim.downloader as api

ft = api.load('fasttext-wiki-news-subwords-300')
topic_model = BERTopic(embedding_model=ft)

## Scikit-Learn Embeddings¶
Scikit-Learn is a framework for more than just machine learning. It offers many preprocessing tools, some of which can be used to create representations for text. Many of these tools are relatively lightweight and do not require a GPU. While the representations may be less expressive than many BERT models, the fact that it runs much faster can make it a relevant candidate to consider.

If you have a scikit-learn compatible pipeline that you'd like to use to embed text then you can also pass this to BERTopic.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from bertopic import BERTopic

pipe = make_pipeline(
    TfidfVectorizer(),
    TruncatedSVD(100)
)

topic_model = BERTopic(embedding_model=pipe)


## OpenAI

To use OpenAI's external API, we need to define our key and explicitly call bertopic.backend.OpenAIBackend to be used in our topic model:

In [ ]:
import openai
from bertopic.backend import OpenAIBackend

client = openai.OpenAI(api_key="sk-...")
embedding_model = OpenAIBackend(client, "text-embedding-ada-002")

topic_model = BERTopic(embedding_model=embedding_model)

## Cohere

To use Cohere's external API, we need to define our key and explicitly call bertopic.backend.CohereBackend to be used in our topic model:

In [5]:
import cohere
from bertopic.backend import CohereBackend

client = cohere.Client("MY_API_KEY")
embedding_model = CohereBackend(client)

topic_model = BERTopic(embedding_model=embedding_model)


## Multimodal

To create embeddings for both text and images in the same vector space, we can use the MultiModalBackend. This model uses a clip-vit based model that is capable of embedding text, images, or both:

In [ ]:
from bertopic.backend import MultiModalBackend
model = MultiModalBackend('clip-ViT-B-32', batch_size=32)

# Embed documents only
doc_embeddings = model.embed_documents(docs)

# Embedding images only
image_embeddings = model.embed_images(images)

# Embed both images and documents, then average them
doc_image_embeddings = model.embed(docs, images)


## Custom Backend

If your backend or model cannot be found in the ones currently available, you can use the bertopic.backend.BaseEmbedder class to create your backend. Below, you will find an example of creating a SentenceTransformer backend for BERTopic:

In [ ]:
from bertopic.backend import BaseEmbedder
from sentence_transformers import SentenceTransformer

class CustomEmbedder(BaseEmbedder):
    def __init__(self, embedding_model):
        super().__init__()
        self.embedding_model = embedding_model

    def embed(self, documents, verbose=False):
        embeddings = self.embedding_model.encode(documents, show_progress_bar=verbose)
        return embeddings 

# Create custom backend
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
custom_embedder = CustomEmbedder(embedding_model=embedding_model)

# Pass custom backend to bertopic
topic_model = BERTopic(embedding_model=custom_embedder)


## Custom Embeddings

The base models in BERTopic are BERT-based models that work well with document similarity tasks. Your documents, however, might be too specific for a general pre-trained model to be used. Fortunately, you can use the embedding model in BERTopic to create document features.

You only need to prepare the document embeddings yourself and pass them through fit_transform of BERTopic:



In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer

# Prepare embeddings
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=False)

# Train our topic model using our pre-trained sentence-transformers embeddings
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs, embeddings)


As you can see above, we used a SentenceTransformer model to create the embedding. You could also have used 🤗 transformers, Doc2Vec, or any other embedding method.

## TF-IDF

As mentioned above, any embedding technique can be used. However, when running UMAP, the typical distance metric is cosine which does not work quite well for a TF-IDF matrix. Instead, BERTopic will recognize that a sparse matrix is passed and use hellinger instead which works quite well for the similarity between probability distributions.

We simply create a TF-IDF matrix and use them as embeddings in our fit_transform method:

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF sparse matrix
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']
vectorizer = TfidfVectorizer(min_df=5)
embeddings = vectorizer.fit_transform(docs)

# Train our topic model using TF-IDF vectors
topic_model = BERTopic(stop_words="english")
topics, probs = topic_model.fit_transform(docs, embeddings)


Here, you will probably notice that creating the embeddings is quite fast whereas fit_transform is quite slow. This is to be expected as reducing the dimensionality of a large sparse matrix takes some time. The inverse of using transformer embeddings is true: creating the embeddings is slow whereas fit_transform is quite fast.